# Step 2 — Hourly Load Profile of the Year

**Purpose.** This notebook turns the connected-load appliance data from Step 1 into a full **8,760-hour annual load profile** (one demand value for every hour of the year) — the same quantity Annex-II and Annex-III compute in `Base-Calculations-00.xlsx`, re-implemented here in Python.

**Method, in brief (from the workbook):**
- Annex-II hand-builds **19 representative "day types"** — a 24-hour pattern of which appliances are switched on, hour by hour, for each season/weekday, each season/weekend, all public holidays, and all festival days. Each hour's value is a "Qty Active" number: how many of that appliance (per household, or per Misc. facility) are actually running in that hour — not just 0 or 1, since e.g. a fridge cycles on/off within an hour (`0.25` = running 25% of the hour) and street lights might have a partial count on.
- Annex-III then assigns **every one of the 365 calendar days of the year** to one of those 19 day types (by season, weekday vs. weekend, and specific public-holiday/festival dates), and stamps out 24 hourly demand values per day — 365 × 24 = 8,760 hours.
- Hourly demand (Wh) for a category = `Σ(Qty Active × Appliance Power) × Household Count`, and the hour's total = the sum across Categories A, B, C, and Misc. — the same formula shape as Step 1's connected-load calculation, just evaluated once per hour instead of once for the whole year.

**A known issue we're fixing here (agreed with the user):** in the source workbook, a ~6-week stretch of weekdays (Aug 3–13 and Sep 16–Oct 30, 2026) accidentally reuses the **April** day-type profile instead of the "Peak Monsoon" and "Late Summer" profiles Annex-II actually built for those periods — almost certainly a copy-paste slip when the calendar was built by hand. We also fix a one-day-early public holiday (labeled "January 16" but applied to January 15). Both fixes are derived directly from the workbook's own data (see §2.2 below) rather than guessed — this notebook extracts the *exact* day-type assignment the source file actually uses for all 365 days, confirms precisely where it diverges from its own stated rules, and patches only those specific days.

**Reference:** see `docs/Workbook-Methodology-Reference.md` §3–4 for the full Annex-II/Annex-III formula breakdown, and `docs/PROJECT_LOG.md` for the fix decisions.

## 2.1 Setup

Same pattern as Step 1: import libraries, locate the project folders, and this time also locate the source Excel workbook — this notebook reads Annex-II and Annex-III **directly from the source file**, rather than re-typing hundreds of hourly values by hand, so it's reproducible and auditable against the original.

In [1]:
import pandas as pd  # pandas: tabular data (DataFrames)
import numpy as np  # numpy: numerical operations
import re  # re: regular expressions, used below to parse Excel formula text
import datetime  # datetime: calendar date arithmetic for building the 365-day year
from pathlib import Path  # Path: OS-independent file paths
import openpyxl  # openpyxl: reads the source .xlsx workbook directly (both cached values and formula text)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()  # find the project's root folder regardless of where the notebook is run from
DATA_DIR = PROJECT_ROOT / "data"  # the shared data/ folder (same one Step 1 used)
DATA_DIR.mkdir(exist_ok=True)  # mkdir(): create it if missing

SOURCE_WORKBOOK = PROJECT_ROOT / "Base-Calculations-00.xlsx"  # the renamed reference workbook, expected alongside the project folder
if not SOURCE_WORKBOOK.exists():
    # fall back to the uploads copy used during development, so this still runs in the cloud sandbox
    SOURCE_WORKBOOK = Path("/mnt/user-data/uploads/Rural Electrification/Base-Calculations.xlsx")

print("pandas:", pd.__version__)
print("Project root:", PROJECT_ROOT.resolve())
print("Source workbook:", SOURCE_WORKBOOK, "  exists:", SOURCE_WORKBOOK.exists())

pandas: 3.0.5
Project root: /home/claude/rural-electrification
Source workbook: /home/claude/rural-electrification/Base-Calculations-00.xlsx   exists: True


## 2.2 Extracting the 19 day-type profiles from Annex-II

Annex-II lays out 19 blocks of 24 rows each — one block per day type (5 winter months + winter weekend, 3 summer months + summer weekend, early monsoon + its weekend, 2 peak-monsoon halves + their shared weekend, late summer + its weekend, all public holidays, all festival days). Each block has the same column layout: a "Qty Active" number per hour for every appliance in Categories A/B/C, and every Misc. load.

We read this straight out of the workbook — the block boundaries (row numbers) and the exact day-type label text — rather than hand-copying values, so this notebook stays correct even if the source workbook's numbers are edited later.

In [2]:
# The 19 day-type blocks in Annex-II: (first row, last row, short name we'll use in this project).
# Row numbers and boundaries confirmed directly against the workbook's own section labels (see markdown above each block in Annex-II).
DAYTYPE_BLOCKS = [
    (10, 33, "November"), (35, 58, "December"), (60, 83, "January"), (85, 108, "February"), (110, 133, "March"),
    (135, 158, "Winter_Weekend"), (161, 184, "April"), (186, 209, "May"), (211, 234, "June_1_15"),
    (236, 259, "Summer_Weekend"), (261, 284, "Early_Monsoon"), (286, 309, "Early_Monsoon_Weekend"),
    (312, 335, "Peak_Monsoon_1"), (337, 360, "Peak_Monsoon_2"), (362, 385, "Peak_Monsoon_Weekend"),
    (387, 410, "Late_Summer"), (412, 435, "Late_Summer_Weekend"), (437, 460, "Public_Holiday"), (462, 485, "Festival_Holiday"),
]
print(f"{len(DAYTYPE_BLOCKS)} day types, {len(DAYTYPE_BLOCKS) * 24} total hourly rows expected")

# Annex-II column index -> (category, appliance/misc-load name), from the sheet's own header row.
DAYTYPE_COLUMNS = {
    3: ("A", "LED Light"), 4: ("A", "Ceiling Fan"), 5: ("A", "Television"), 6: ("A", "Iron"), 7: ("A", "Refrigerator"), 8: ("A", "Washing Machine"),
    11: ("B", "LED Light"), 12: ("B", "Ceiling Fan"), 13: ("B", "Television"), 14: ("B", "Iron"), 15: ("B", "Refrigerator"),
    18: ("C", "LED Light"), 19: ("C", "Ceiling Fan"),
    22: ("Misc", "Hospital"), 23: ("Misc", "Street Light"), 24: ("Misc", "Cold Storage"), 25: ("Misc", "School"), 26: ("Misc", "Others"),
}

19 day types, 456 total hourly rows expected


In [3]:
def extract_daytype_profiles(workbook_path) -> pd.DataFrame:
    """
    Read all 19 day-type blocks from Annex-II and return one tidy (long-format) table:
    one row per (day_type, hour_of_day, category, item), with its Qty Active value.
    hour_of_day runs 0-23 (0 = the block's first row, i.e. the first hour of that day type).
    """
    wb = openpyxl.load_workbook(workbook_path, data_only=True, read_only=True)  # data_only=True: read cached numbers, not formula text; read_only=True: much faster/lighter for a big workbook
    ws = wb["Annex-II"]

    # Build a lookup: Annex-II row number -> (day_type name, hour_of_day 0-23)
    row_to_daytype_hour = {}  # dict: maps every row across all 19 blocks to which day type + hour it belongs to
    for block_start, block_end, day_type in DAYTYPE_BLOCKS:
        for row in range(block_start, block_end + 1):
            row_to_daytype_hour[row] = (day_type, row - block_start)

    min_row = min(b[0] for b in DAYTYPE_BLOCKS)  # the first row across all blocks, so we scan the whole span in one pass
    max_row = max(b[1] for b in DAYTYPE_BLOCKS)  # the last row across all blocks

    records = []  # collect one dict per (day_type, hour, category, item) as we scan
    for row_cells in ws.iter_rows(min_row=min_row, max_row=max_row, min_col=1, max_col=26):  # iter_rows(): stream through the rows once, sequentially (fast even in read_only mode)
        row_num = row_cells[0].row
        if row_num not in row_to_daytype_hour:
            continue  # skip blank/label rows between blocks
        day_type, hour_of_day = row_to_daytype_hour[row_num]
        for col_idx, (category, item) in DAYTYPE_COLUMNS.items():
            qty_active = row_cells[col_idx - 1].value  # -1: openpyxl columns are 1-indexed, list is 0-indexed
            records.append({"day_type": day_type, "hour_of_day": hour_of_day,
                             "category": category, "item": item, "qty_active": qty_active})

    wb.close()  # close the workbook to free memory
    return pd.DataFrame(records)


daytype_profiles = extract_daytype_profiles(SOURCE_WORKBOOK)  # call the function on our source workbook
print(f"Extracted {len(daytype_profiles):,} rows "
      f"(expected {len(DAYTYPE_BLOCKS)} day types x 24 hours x {len(DAYTYPE_COLUMNS)} items = {len(DAYTYPE_BLOCKS)*24*len(DAYTYPE_COLUMNS):,})")
daytype_profiles.head(18)

Extracted 8,208 rows (expected 19 day types x 24 hours x 18 items = 8,208)


,day_type,hour_of_day,category,item,qty_active
0,November,0,A,LED Light,1.00
1,November,0,A,Ceiling Fan,1.00
2,November,0,A,Television,0.00
3,November,0,A,Iron,0.00
4,November,0,A,Refrigerator,0.25
5,November,0,A,Washing Machine,0.00
6,November,0,B,LED Light,0.00
7,November,0,B,Ceiling Fan,1.00
8,November,0,B,Television,0.00
9,November,0,B,Iron,0.00


### Sanity check

Spot-check one specific hour we already know the answer to from the methodology audit: **November, hour 0** (midnight–1am) should show Category A's Refrigerator cycling at `0.25` (running a quarter of the hour) and 40 street lights on.

In [4]:
check = daytype_profiles[(daytype_profiles["day_type"] == "November") & (daytype_profiles["hour_of_day"] == 0)]  # filter to the one hour we're spot-checking
display(check)  # display(): show the filtered table (works like the bare-variable auto-display, but usable mid-cell)

fridge_qty = check.loc[(check["category"] == "A") & (check["item"] == "Refrigerator"), "qty_active"].iloc[0]  # .iloc[0]: pull the single matching value out
streetlight_qty = check.loc[(check["category"] == "Misc") & (check["item"] == "Street Light"), "qty_active"].iloc[0]
assert fridge_qty == 0.25, f"Expected fridge qty_active=0.25, got {fridge_qty}"  # assert: stop if either check fails
assert streetlight_qty == 40, f"Expected street light qty_active=40, got {streetlight_qty}"
print("✓ Matches the known workbook values for November, hour 0")

,day_type,hour_of_day,category,item,qty_active
0,November,0,A,LED Light,1.00
1,November,0,A,Ceiling Fan,1.00
2,November,0,A,Television,0.00
3,November,0,A,Iron,0.00
4,November,0,A,Refrigerator,0.25
5,November,0,A,Washing Machine,0.00
6,November,0,B,LED Light,0.00
7,November,0,B,Ceiling Fan,1.00
8,November,0,B,Television,0.00
9,November,0,B,Iron,0.00


✓ Matches the known workbook values for November, hour 0


### Saving the day-type profile library

Saved as CSV, same pattern as Step 1 — this becomes the default data the app reads, and (like the appliance tables) is meant to be user-editable later: download as Excel, tweak the "Qty Active" pattern for any day type, re-upload.

In [5]:
daytype_profiles.to_csv(DATA_DIR / "default_daytype_profiles.csv", index=False)  # to_csv(): save the full day-type library to disk
print("Saved to:", (DATA_DIR / "default_daytype_profiles.csv").resolve())

Saved to: /home/claude/rural-electrification/data/default_daytype_profiles.csv


## 2.3 The 365-day calendar: which day type applies to each day

Annex-III assigns one of the 19 Annex-II day types to every calendar day of 2026, via an Excel array formula that points at a specific row-range of Annex-II for that day (e.g. `INDEX('Annex-II'!C$60:'Annex-II'!C$83, ...)` — the January weekday block). Rather than re-deriving the calendar rules from scratch (season start/end dates, which would risk introducing *new*, different assumptions from the workbook's own), we read the **exact row-range each day's formula already points to**, straight out of the source file. This gives us the day-type assignment exactly as originally implemented, bugs included — which we can then audit and patch precisely.

In [6]:
# Map each Annex-II row-range back to its day-type name (reusing DAYTYPE_BLOCKS from §2.2)
ROWRANGE_TO_DAYTYPE = {(start, end): name for start, end, name in DAYTYPE_BLOCKS}  # dict: (row_start, row_end) -> day type name, for looking up which block a formula points to

ARRAYFORMULA_ROW_PATTERN = re.compile(r"\$(\d+):'Annex-II'!\$?[A-Z]+\$(\d+)")  # regex: pulls the two row numbers out of an INDEX() formula referencing Annex-II


def extract_asimplemented_calendar(workbook_path, year: int = 2026, n_days: int = 365) -> pd.DataFrame:
    """
    Read, for every day of the year, which Annex-II day-type block Annex-III's own formulas
    actually reference — i.e. the calendar exactly as implemented in the source workbook,
    bugs and all. Returns one row per day: day number, date, weekday name, and day_type.
    """
    wb = openpyxl.load_workbook(workbook_path, data_only=False, read_only=True)  # data_only=False: we need the formula TEXT this time, not a cached number
    ws = wb["Annex-III"]

    first_row = 10  # Annex-III row 10 = Day 1, Hour 1 (confirmed against the sheet's own Date/Hour columns)
    last_row = first_row + n_days * 24 - 1  # the last row of the last day's 24 hours

    records = []  # collect one dict per day as we scan
    row_counter = 0  # counts every row we pass, so we know when we've hit the first hour of a new day (every 24th row)
    day = 0
    for row_cells in ws.iter_rows(min_row=first_row, max_row=last_row, min_col=7, max_col=7):  # column G: Category A's first appliance, whose formula reveals the day-type block for this hour
        if row_counter % 24 == 0:  # only the first hour of each day is needed — all 24 hours of a day share the same block
            day += 1
            array_formula = row_cells[0].value
            formula_text = getattr(array_formula, "text", str(array_formula))  # ArrayFormula objects store their text on .text
            match = ARRAYFORMULA_ROW_PATTERN.search(formula_text)
            if not match:
                raise ValueError(f"Could not parse day-type reference on day {day}: {formula_text!r}")
            row_start, row_end = int(match.group(1)), int(match.group(2))
            day_type = ROWRANGE_TO_DAYTYPE.get((row_start, row_end), f"UNKNOWN({row_start},{row_end})")
            date = datetime.date(year, 1, 1) + datetime.timedelta(days=day - 1)  # convert day number (1-365) to an actual calendar date
            records.append({"day": day, "date": date, "weekday": date.strftime("%A"), "day_type": day_type})
        row_counter += 1

    wb.close()
    return pd.DataFrame(records)


asimplemented_calendar = extract_asimplemented_calendar(SOURCE_WORKBOOK)  # call the function on our source workbook
print(f"Extracted day-type assignment for {len(asimplemented_calendar)} days")
asimplemented_calendar.head(3)

Extracted day-type assignment for 365 days


,day,date,weekday,day_type
0,1,2026-01-01,Thursday,January
1,2,2026-01-02,Friday,January
2,3,2026-01-03,Saturday,Winter_Weekend


### Auditing the "April" day type — where is it legitimately April, and where is it a bug?

Annex-II built dedicated "Peak Monsoon" (two halves) and "Late Summer" day types specifically for August–October. If the calendar were implemented correctly, "April" should only ever appear on actual April weekdays. Any other month using it is the copy-paste bug.

In [7]:
april_block_days = asimplemented_calendar[asimplemented_calendar["day_type"] == "April"].copy()  # filter: every day currently assigned the "April" day type
april_block_days["month"] = april_block_days["date"].apply(lambda d: d.month)  # extract the calendar month, to separate legitimate April from the bug

legit_april = april_block_days[april_block_days["month"] == 4]  # rows actually in April — these are correct, not a bug
bug_days = april_block_days[april_block_days["month"] != 4]  # rows NOT in April — these are the copy-paste bug

print(f"'April' day type used on {len(april_block_days)} days total:")
print(f"  - {len(legit_april)} days are genuinely in April (correct)")
print(f"  - {len(bug_days)} days are in other months (the bug) — dates: "
      f"{bug_days['date'].min()} to {bug_days['date'].max()}")
bug_days[["date", "weekday"]]

'April' day type used on 64 days total:
  - 22 days are genuinely in April (correct)
  - 42 days are in other months (the bug) — dates: 2026-08-03 to 2026-10-30


,date,weekday
214,2026-08-03,Monday
215,2026-08-04,Tuesday
216,2026-08-05,Wednesday
217,2026-08-06,Thursday
218,2026-08-07,Friday
221,2026-08-10,Monday
222,2026-08-11,Tuesday
223,2026-08-12,Wednesday
224,2026-08-13,Thursday
258,2026-09-16,Wednesday


### Applying the fix

Per Annex-II's own labels: **August 1–15** should use the "Peak Monsoon (June 16 – August 15)" day type, and **September 16 – October 31** should use "Late Summer (September 15 – October 31)". We only touch the specific buggy days identified above — every other day (including the weekends in these same months, which already correctly used the Peak-Monsoon/Late-Summer *weekend* day types) is left exactly as implemented.

In [8]:
def fix_calendar(calendar_df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply the two agreed fixes to the as-implemented calendar:
      1. 'April' day type wrongly used outside April -> replace with the correct
         Peak_Monsoon_1 (Aug) or Late_Summer (Sep16-Oct31) day type.
      2. The public holiday labeled 'January 16' was applied to January 15 instead -> swap them.
    Returns a new DataFrame with an extra 'original_day_type' column, so every change is traceable.
    """
    fixed = calendar_df.copy()  # copy(): work on a copy so the original as-implemented table is preserved for comparison
    fixed["original_day_type"] = fixed["day_type"]  # keep a record of what each day was before any fix

    is_april_block = fixed["day_type"] == "April"  # boolean mask: every day currently using the April block
    month = fixed["date"].apply(lambda d: d.month)  # the calendar month of each date
    day_of_month = fixed["date"].apply(lambda d: d.day)  # the day-of-month of each date

    aug_bug = is_april_block & (month == 8) & (day_of_month <= 15)  # the Aug 1-15 mis-assigned days
    fixed.loc[aug_bug, "day_type"] = "Peak_Monsoon_1"  # .loc[]: overwrite just those rows' day_type

    sep_oct_bug = is_april_block & (((month == 9) & (day_of_month >= 16)) | (month == 10))  # the Sep16-Oct31 mis-assigned days
    fixed.loc[sep_oct_bug, "day_type"] = "Late_Summer"

    jan15_bug = (fixed["date"] == datetime.date(2026, 1, 15))  # the one-day-early holiday
    jan16 = (fixed["date"] == datetime.date(2026, 1, 16))
    fixed.loc[jan15_bug, "day_type"] = "January"  # Jan 15 reverts to its natural (non-holiday) weekday assignment
    fixed.loc[jan16, "day_type"] = "Public_Holiday"  # Jan 16 becomes the holiday, matching Annex-II's own label

    fixed["was_fixed"] = fixed["day_type"] != fixed["original_day_type"]  # flag every row that actually changed
    return fixed


corrected_calendar = fix_calendar(asimplemented_calendar)  # call: apply the fix
n_fixed = corrected_calendar["was_fixed"].sum()  # sum(): count how many days changed
print(f"{n_fixed} of {len(corrected_calendar)} days changed by the fix")

44 of 365 days changed by the fix


### Before/after: every day that changed, and why

In [9]:
changed = corrected_calendar[corrected_calendar["was_fixed"]][["date", "weekday", "original_day_type", "day_type"]]  # select just the changed rows and the columns worth showing
print(f"{len(changed)} days changed:")
changed

44 days changed:


,date,weekday,original_day_type,day_type
14,2026-01-15,Thursday,Public_Holiday,January
15,2026-01-16,Friday,January,Public_Holiday
214,2026-08-03,Monday,April,Peak_Monsoon_1
215,2026-08-04,Tuesday,April,Peak_Monsoon_1
216,2026-08-05,Wednesday,April,Peak_Monsoon_1
217,2026-08-06,Thursday,April,Peak_Monsoon_1
218,2026-08-07,Friday,April,Peak_Monsoon_1
221,2026-08-10,Monday,April,Peak_Monsoon_1
222,2026-08-11,Tuesday,April,Peak_Monsoon_1
223,2026-08-12,Wednesday,April,Peak_Monsoon_1


### Sanity checks on the fix

In [10]:
# 1. No day outside April should use the "April" day type anymore
remaining_april = corrected_calendar[corrected_calendar["day_type"] == "April"]
assert (remaining_april["date"].apply(lambda d: d.month) == 4).all(), "Some non-April day still uses the April block!"
print(f"✓ 'April' day type now used only in April ({len(remaining_april)} days)")

# 2. Exactly one Public Holiday moved from Jan 15 to Jan 16
assert corrected_calendar.loc[corrected_calendar["date"] == datetime.date(2026, 1, 15), "day_type"].iloc[0] == "January"
assert corrected_calendar.loc[corrected_calendar["date"] == datetime.date(2026, 1, 16), "day_type"].iloc[0] == "Public_Holiday"
print("✓ January 16 (not 15) is now the public holiday")

# 3. Every day of the year still has exactly one assignment (no duplicates/gaps)
assert len(corrected_calendar) == 365
assert corrected_calendar["date"].nunique() == 365
print("✓ All 365 days accounted for, no duplicates")

# 4. Total fixed days matches what we found in the audit (9 August days + 33 Sep-Oct days + 2 holiday-swap days)
assert n_fixed == len(bug_days) + 2, f"Expected {len(bug_days) + 2} changed days, got {n_fixed}"
print(f"✓ {n_fixed} days changed, matching the audit (({len(bug_days)} calendar-mapping days) + (2 holiday-swap days))")

✓ 'April' day type now used only in April (22 days)
✓ January 16 (not 15) is now the public holiday
✓ All 365 days accounted for, no duplicates
✓ 44 days changed, matching the audit ((42 calendar-mapping days) + (2 holiday-swap days))


### Saving the corrected calendar

In [11]:
corrected_calendar.to_csv(DATA_DIR / "annual_calendar_2026.csv", index=False)  # to_csv(): save the final day->day-type mapping for the year
print("Saved to:", (DATA_DIR / "annual_calendar_2026.csv").resolve())

Saved to: /home/claude/rural-electrification/data/annual_calendar_2026.csv


## 2.4 Computing the full 8,760-hour demand profile

Now we combine everything: for every hour of the year, look up its day type (§2.3), pull that day type's "Qty Active" pattern (§2.2), and multiply by each appliance's power rating (from **Step 1**'s appliance tables) — the same `Qty Active × Power` formula Annex-II itself uses.

One detail worth being explicit about, since it's easy to assume otherwise: **"Qty Active" is not a fraction of the appliances a household owns — it already IS a count of units running.** E.g. Category A owns 5 LED lights per house (Step 1's `qty_per_house`); if Qty Active = 1 at some hour, that means "1 of those 5 lights is on", not "20% of some single light". Annex-II's own formula confirms this: it multiplies Qty Active directly by the appliance's **power rating**, never by `qty_per_house` — `qty_per_house` only matters for Step 1's connected/peak load, not for the hourly usage pattern. Practically, this means the default hourly *usage pattern* (how many lights tend to be on) is an independent assumption from *ownership count*, and won't automatically rescale if `qty_per_house` is edited later — worth keeping in mind when editing either table.

For the same reason, our new **"Others" placeholder appliance (Step 1, §1.2) has no hourly usage pattern defined yet** — Annex-II never modeled it (it didn't exist in the original workbook), so until a usage pattern is added for it, it contributes only to connected/peak load, not to the hourly profile. Flagged here as a known follow-up rather than silently ignored.

In [12]:
# Load Step 1's default household/appliance data (the same tables 01_Connected_Load_Estimation.ipynb built and saved)
household_categories = pd.read_csv(DATA_DIR / "default_household_categories.csv")  # read_csv(): load the household counts per category
appliances = pd.read_csv(DATA_DIR / "default_appliances.csv")  # load the per-category appliance list (power, qty per house)
misc_loads = pd.read_csv(DATA_DIR / "default_misc_loads.csv")  # load the Misc./community load list

print(f"Loaded {len(household_categories)} household categories, {len(appliances)} appliance lines, {len(misc_loads)} misc. load lines")

Loaded 3 household categories, 16 appliance lines, 5 misc. load lines


In [13]:
# Build one combined "power lookup" table: (category, item) -> power_w, covering both household appliances and Misc. loads
appliance_power = appliances[["category", "appliance", "power_w"]].rename(columns={"appliance": "item"})  # rename to "item" so it matches daytype_profiles' column name
misc_power = misc_loads[["appliance", "power_w"]].rename(columns={"appliance": "item"})  # same rename for Misc.
misc_power.insert(0, "category", "Misc")  # insert(): tag every Misc. row with category="Misc", so it can stack with appliance_power

power_lookup = pd.concat([appliance_power, misc_power], ignore_index=True)  # concat(): combine into one lookup table
power_lookup

,category,item,power_w
0,A,LED Light,18
1,A,Ceiling Fan,50
2,A,Television,60
3,A,Iron,800
4,A,Refrigerator,800
5,A,Washing Machine,350
6,B,LED Light,18
7,B,Ceiling Fan,50
8,B,Television,60
9,B,Iron,800


In [14]:
def build_hour_skeleton(n_days: int = 365) -> pd.DataFrame:
    """Build the 8,760-row hour-of-year skeleton: hour_of_year (1-8760), day (1-365), hour_of_day (0-23)."""
    hour_of_year = np.arange(1, n_days * 24 + 1)  # arange(): 1, 2, 3, ... 8760
    day = (hour_of_year - 1) // 24 + 1  # integer division: which day (1-365) each hour falls in
    hour_of_day = (hour_of_year - 1) % 24  # modulo: which hour within that day (0-23)
    return pd.DataFrame({"hour_of_year": hour_of_year, "day": day, "hour_of_day": hour_of_day})


hour_skeleton = build_hour_skeleton()  # call: build the empty 8,760-row skeleton
hour_skeleton = hour_skeleton.merge(corrected_calendar[["day", "date", "day_type"]], on="day", how="left")  # merge(): attach each hour's date and (fixed) day type from §2.3
print(f"{len(hour_skeleton):,} hours built")
hour_skeleton.head(3)

8,760 hours built


,hour_of_year,day,hour_of_day,date,day_type
0,1,1,0,2026-01-01,January
1,2,1,1,2026-01-01,January
2,3,1,2,2026-01-01,January


In [15]:
def compute_hourly_demand(hour_skeleton_df: pd.DataFrame,
                           daytype_profiles_df: pd.DataFrame,
                           power_lookup_df: pd.DataFrame,
                           household_categories_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute Wh demand for every hour of the year, broken down by category (A/B/C/Misc) and totaled.
    Returns hour_skeleton_df with four new columns: A_wh, B_wh, C_wh, Misc_wh, total_wh.
    """
    # Expand each hour into one row per (category, item) line, by matching its day type + hour of day
    # against the day-type profile library — this is the "Qty Active" lookup.
    expanded = hour_skeleton_df.merge(daytype_profiles_df, on=["day_type", "hour_of_day"], how="left")  # merge(): 8,760 hours x ~18 items/hour = ~157,680 rows

    expanded = expanded.merge(power_lookup_df, on=["category", "item"], how="left")  # merge(): attach each line's power rating (W)
    expanded = expanded.merge(household_categories_df[["category", "household_count"]], on="category", how="left")  # merge(): attach household_count (only meaningful for A/B/C)
    expanded["household_count"] = expanded["household_count"].fillna(1)  # fillna(): Misc. lines have no household multiplier, so treat as x1

    expanded["wh"] = expanded["qty_active"] * expanded["power_w"] * expanded["household_count"]  # compute: Qty Active x Power (W) x Household count = Wh for this line, this hour

    by_category = (
        expanded.groupby(["hour_of_year", "category"])["wh"]  # groupby(): sum every line's Wh within the same hour and category
        .sum()
        .unstack("category")  # unstack(): pivot category from rows into columns (A_wh, B_wh, C_wh, Misc_wh side by side)
        .fillna(0)  # fillna(): an hour with no Misc. activity, etc. still gets an explicit 0 rather than a gap
    )
    by_category.columns = [f"{c}_wh" for c in by_category.columns]  # rename columns e.g. "A" -> "A_wh"
    by_category["total_wh"] = by_category.sum(axis=1)  # sum across columns: total demand for that hour, all categories combined

    return hour_skeleton_df.merge(by_category, on="hour_of_year")  # merge(): attach the computed columns back onto the hour skeleton


hourly_profile = compute_hourly_demand(hour_skeleton, daytype_profiles, power_lookup, household_categories)  # call the function
print(f"Computed {len(hourly_profile):,} hourly demand values")
hourly_profile.head(3)

Computed 8,760 hourly demand values


,hour_of_year,day,hour_of_day,date,day_type,A_wh,B_wh,C_wh,Misc_wh,total_wh
0,1,1,0,2026-01-01,January,40200.0,185000.0,0.0,36100.0,261300.0
1,2,1,1,2026-01-01,January,40200.0,185000.0,0.0,36100.0,261300.0
2,3,1,2,2026-01-01,January,40200.0,185000.0,0.0,57100.0,282300.0


### Sanity checks

1. **Hour 1 (Jan 1, midnight–1am)** is unaffected by either fix — cross-check it directly against Annex-III's own cached value for that hour.
2. **Annual total** — compare against the workbook's 3.4726 GWh. We expect a *small* difference, not a match: our total now correctly uses the Peak-Monsoon/Late-Summer profiles for those 42 days instead of April's, so the two totals are supposed to diverge slightly.

In [16]:
hour1_wh = hourly_profile.loc[hourly_profile["hour_of_year"] == 1, "total_wh"].iloc[0]  # pull out our computed value for hour 1
workbook_hour1_wh = 261300  # Annex-III row 10, column AF — read directly from the source workbook
print(f"Hour 1 (Jan 1, 00:00-01:00): computed = {hour1_wh:,.0f} Wh, workbook = {workbook_hour1_wh:,.0f} Wh")
assert hour1_wh == workbook_hour1_wh, "Hour 1 should match the workbook exactly (this hour isn't affected by either fix)"
print("✓ Exact match")

Hour 1 (Jan 1, 00:00-01:00): computed = 261,300 Wh, workbook = 261,300 Wh
✓ Exact match


In [17]:
annual_total_gwh = hourly_profile["total_wh"].sum() / 1e9  # convert Wh to GWh
workbook_annual_total_gwh = 3.472586398  # Annex-III!AG10, the workbook's own annual total (with the calendar bug still in it)
diff_pct = (annual_total_gwh - workbook_annual_total_gwh) / workbook_annual_total_gwh * 100

print(f"Computed annual total (fixed calendar):  {annual_total_gwh:.6f} GWh")
print(f"Workbook annual total (buggy calendar):  {workbook_annual_total_gwh:.6f} GWh")
print(f"Difference: {annual_total_gwh - workbook_annual_total_gwh:+.6f} GWh ({diff_pct:+.3f}%)")

Computed annual total (fixed calendar):  3.437285 GWh
Workbook annual total (buggy calendar):  3.472586 GWh
Difference: -0.035301 GWh (-1.017%)


The fixed total comes out **lower**, not higher — worth checking why rather than assuming a direction. Comparing the per-day totals of the day types involved answers it directly: in this workbook's own assumptions, the "Peak Monsoon" and "Late Summer" profiles carry *less* daily demand than "April" (less fan/cooling and lighting activity assumed during the cooler, more humid monsoon months than during April's hot pre-monsoon peak) — so replacing April with the seasonally-correct profiles on those 42 days reduces the annual total, rather than increasing it.

In [18]:
def daytype_total_wh(day_type_name: str) -> float:
    """Total Wh across ALL households/misc. loads for one full day of a given day type (a quick way to compare day types)."""
    lines = daytype_profiles[daytype_profiles["day_type"] == day_type_name].merge(power_lookup, on=["category", "item"])  # merge(): attach power ratings
    lines = lines.merge(household_categories[["category", "household_count"]], on="category", how="left")  # merge(): attach household counts
    lines["household_count"] = lines["household_count"].fillna(1)  # Misc. lines: no household multiplier
    return (lines["qty_active"] * lines["power_w"] * lines["household_count"]).sum()  # compute and sum every line's Wh


for name in ["April", "Peak_Monsoon_1", "Peak_Monsoon_2", "Late_Summer"]:  # for loop: compare the day types involved in the fix
    print(f"{name:20s} {daytype_total_wh(name):>14,.0f} Wh/day")

April                    10,002,655 Wh/day
Peak_Monsoon_1            7,988,755 Wh/day
Peak_Monsoon_2            7,988,755 Wh/day
Late_Summer               9,497,755 Wh/day


### Independent hand-check on one of the *fixed* hours

Since the workbook itself is wrong on the fixed days, we can't cross-check those against it. Instead, verify our pipeline's output for one fixed hour (Aug 3, hour 0 — now `Peak_Monsoon_1`) against a fully independent, hand-written calculation from the raw day-type table.

In [19]:
aug3_hour0 = hourly_profile[(hourly_profile["date"] == datetime.date(2026, 8, 3)) & (hourly_profile["hour_of_day"] == 0)]  # find that specific hour in our computed profile
pipeline_total_wh = aug3_hour0["total_wh"].iloc[0]

# Hand-calculation: manually pull Peak_Monsoon_1, hour 0's Qty Active values and multiply out by hand
pm1_hour0 = daytype_profiles[(daytype_profiles["day_type"] == "Peak_Monsoon_1") & (daytype_profiles["hour_of_day"] == 0)]
hand_total_wh = 0.0
for _, line in pm1_hour0.iterrows():  # for loop: walk through each appliance/misc line for this day type + hour by hand
    power = power_lookup.loc[(power_lookup["category"] == line["category"]) & (power_lookup["item"] == line["item"]), "power_w"].iloc[0]
    if line["category"] == "Misc":
        hand_total_wh += line["qty_active"] * power  # Misc: no household multiplier
    else:
        hh_count = household_categories.loc[household_categories["category"] == line["category"], "household_count"].iloc[0]
        hand_total_wh += line["qty_active"] * power * hh_count  # household categories: multiply by household count

print(f"Pipeline result:      {pipeline_total_wh:,.0f} Wh")
print(f"Independent hand-calc: {hand_total_wh:,.0f} Wh")
assert pipeline_total_wh == hand_total_wh, "Pipeline and hand-calculation should match exactly"
print("✓ Exact match — the merge/groupby pipeline is computing the same thing a manual line-by-line calculation would")

Pipeline result:      232,500 Wh
Independent hand-calc: 232,500 Wh
✓ Exact match — the merge/groupby pipeline is computing the same thing a manual line-by-line calculation would


## 2.5 Saving the annual hourly profile

The full 8,760-row profile — one row per hour, with per-category and total Wh demand — is the main output of this notebook. It becomes the input to Step 4 (solar sizing) and Step 3 (weekly/monthly/yearly visualization).

In [20]:
hourly_profile.to_csv(DATA_DIR / "hourly_load_profile_2026.csv", index=False)  # to_csv(): save the full annual hourly profile
print("Saved to:", (DATA_DIR / "hourly_load_profile_2026.csv").resolve())
print(f"Shape: {hourly_profile.shape[0]:,} rows x {hourly_profile.shape[1]} columns")
hourly_profile[["hour_of_year", "date", "hour_of_day", "day_type", "A_wh", "B_wh", "C_wh", "Misc_wh", "total_wh"]].head(5)

Saved to: /home/claude/rural-electrification/data/hourly_load_profile_2026.csv
Shape: 8,760 rows x 10 columns


,hour_of_year,date,hour_of_day,day_type,A_wh,B_wh,C_wh,Misc_wh,total_wh
0,1,2026-01-01,0,January,40200.0,185000.0,0.0,36100.0,261300.0
1,2,2026-01-01,1,January,40200.0,185000.0,0.0,36100.0,261300.0
2,3,2026-01-01,2,January,40200.0,185000.0,0.0,57100.0,282300.0
3,4,2026-01-01,3,January,40200.0,185000.0,0.0,57100.0,282300.0
4,5,2026-01-01,4,January,40200.0,185000.0,0.0,56200.0,281400.0


## 2.6 Excel download / edit / re-upload — the day-type profile table

Same pattern as Step 1: export the current day-type "Qty Active" patterns to a formatted, editable `.xlsx`, and re-import a user-edited copy, with validation. Since the day-type table is naturally a grid (19 day types x 24 hours x 18 appliance/misc items), we pivot it to a **wide** layout for export — one row per (day type, hour), one column per appliance/misc item — much easier to look at and edit in Excel than the tidy/long format we've been computing with.

In [21]:
def export_daytype_template(daytype_profiles_df: pd.DataFrame, output_path) -> None:
    """Write the day-type Qty-Active table to a formatted .xlsx, pivoted wide (one column per item) for easy editing."""
    wide = daytype_profiles_df.pivot_table(  # pivot_table(): reshape from long (one row per value) to wide (one row per hour, one column per item)
        index=["day_type", "hour_of_day"], columns=["category", "item"], values="qty_active"
    )
    wide.columns = [f"{cat}|{item}" for cat, item in wide.columns]  # flatten the two-level column headers into single strings like "A|LED Light"
    wide = wide.reset_index()  # reset_index(): turn day_type/hour_of_day back into normal columns instead of a row index

    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:  # ExcelWriter(): open a new .xlsx for writing, with formatting support
        workbook = writer.book
        header_fmt = workbook.add_format({"bold": True, "bg_color": "#2a78d6", "font_color": "white", "border": 1})  # same header style as Step 1's template

        instructions = pd.DataFrame({"Instructions": [
            "Each row is one hour of one day type. Edit any 'Qty Active' value (a category|item column).",
            "Qty Active is a COUNT of units running that hour (e.g. how many of the household's lights are on), not a fraction.",
            "Do not add/remove/rename day_type or hour_of_day rows, or rename columns — the app matches them by name on re-upload.",
            "Save the file, then re-upload it in the app to use these values instead of the defaults.",
        ]})
        instructions.to_excel(writer, sheet_name="Instructions", index=False)
        wide.to_excel(writer, sheet_name="Day Type Profiles", index=False)  # to_excel(): write the pivoted table to its own sheet

        for sheet_name, df in [("Instructions", instructions), ("Day Type Profiles", wide)]:  # for loop: bold headers + sized columns on every sheet, same as Step 1
            ws = writer.sheets[sheet_name]
            for col_idx, col_name in enumerate(df.columns):
                ws.write(0, col_idx, col_name, header_fmt)
                width = max(10, min(28, len(str(col_name)) + 2))
                ws.set_column(col_idx, col_idx, width)


def import_daytype_template(input_path) -> dict:
    """Read a workbook in the export_daytype_template() shape back into a tidy (long-format) DataFrame, with validation."""
    errors = []
    sheets = pd.read_excel(input_path, sheet_name=None)  # read_excel(): read every sheet at once

    if "Day Type Profiles" not in sheets:
        return {"daytype_profiles": None, "errors": ["Missing required sheet: 'Day Type Profiles'"]}

    wide = sheets["Day Type Profiles"]
    required_index_cols = ["day_type", "hour_of_day"]
    missing = [c for c in required_index_cols if c not in wide.columns]
    if missing:
        errors.append(f"'Day Type Profiles' sheet is missing column(s): {missing}")

    if not errors:
        item_cols = [c for c in wide.columns if c not in required_index_cols]  # every other column is a "category|item" Qty-Active column
        long = wide.melt(id_vars=required_index_cols, value_vars=item_cols,  # melt(): the inverse of pivot_table — back to one row per (day_type, hour, item)
                          var_name="category_item", value_name="qty_active")
        long[["category", "item"]] = long["category_item"].str.split("|", n=1, expand=True)  # split "A|LED Light" back into two columns
        long = long.drop(columns="category_item")

        if len(wide) != 19 * 24:
            errors.append(f"Expected {19*24} rows (19 day types x 24 hours), found {len(wide)}")
        if (long["qty_active"] < 0).any():
            errors.append("Some Qty Active value(s) are negative, which isn't physically valid")

    if errors:
        return {"daytype_profiles": None, "errors": errors}
    return {"daytype_profiles": long[["day_type", "hour_of_day", "category", "item", "qty_active"]], "errors": []}

In [22]:
daytype_template_path = DATA_DIR / "Day_Type_Profiles_Template.xlsx"
export_daytype_template(daytype_profiles, daytype_template_path)  # call: write the current day-type profiles out as an editable template
print("Exported to:", daytype_template_path.resolve())

reimported_daytypes = import_daytype_template(daytype_template_path)  # call: immediately re-import, to test the round trip
if reimported_daytypes["errors"]:
    print("Validation errors:", reimported_daytypes["errors"])
else:
    # Sort both tables the same way before comparing, since melt()/pivot_table() don't guarantee row order
    left = daytype_profiles.sort_values(["day_type", "hour_of_day", "category", "item"]).reset_index(drop=True)
    right = reimported_daytypes["daytype_profiles"].sort_values(["day_type", "hour_of_day", "category", "item"]).reset_index(drop=True)
    pd.testing.assert_frame_equal(left, right, check_dtype=False)  # assert_frame_equal(): raises if the two tables differ in any value
    print("✓ Round-trip export -> re-import reproduces the original day-type table exactly")

Exported to: /home/claude/rural-electrification/data/Day_Type_Profiles_Template.xlsx
✓ Round-trip export -> re-import reproduces the original day-type table exactly


## 2.7 Alternative: upload a complete custom hourly profile

Some users will have real metered/estimated hourly demand data already, rather than wanting to build it up from appliances and day types at all. For that case: a downloadable blank template with just **Date, Hour, Load (kWh)** columns, and an importer that validates a filled-in version and — if it passes — can be used **in place of** the computed profile from §2.4 for everything downstream (visualization, solar sizing, etc.).

In [23]:
def export_hourly_profile_template(output_path, year: int = 2026) -> None:
    """Write a blank Date/Hour/Load template — 8,760 rows, ready for the user to fill in their own hourly demand data."""
    dates = pd.date_range(f"{year}-01-01", periods=365, freq="D")  # date_range(): every calendar date in the year
    template_rows = []
    for date in dates:  # for loop: one row per hour of every day
        for hour in range(24):
            template_rows.append({"Date": date.strftime("%Y-%m-%d"), "Hour": hour, "Load_kWh": None})  # Load_kWh left blank for the user to fill in
    template = pd.DataFrame(template_rows)

    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        workbook = writer.book
        header_fmt = workbook.add_format({"bold": True, "bg_color": "#2a78d6", "font_color": "white", "border": 1})

        instructions = pd.DataFrame({"Instructions": [
            "Fill in Load_kWh for every hour with your own hourly demand data (kWh consumed in that hour).",
            "Hour runs 0-23, where 0 means the hour from midnight to 1am, 23 means 11pm to midnight.",
            "Do not add, remove, or reorder rows, and do not rename columns.",
            "Save the file, then upload it in the app to use this data instead of the computed profile.",
        ]})
        instructions.to_excel(writer, sheet_name="Instructions", index=False)
        template.to_excel(writer, sheet_name="Hourly Profile", index=False)

        for sheet_name, df in [("Instructions", instructions), ("Hourly Profile", template)]:
            ws = writer.sheets[sheet_name]
            for col_idx, col_name in enumerate(df.columns):
                ws.write(0, col_idx, col_name, header_fmt)
                width = max(12, min(40, len(str(col_name)) + 4))
                ws.set_column(col_idx, col_idx, width)


def import_hourly_profile(input_path, expected_year: int = 2026) -> dict:
    """Read a user-filled Date/Hour/Load_kWh workbook, validate it, and return an 8,760-row profile ready to use."""
    errors = []
    df = pd.read_excel(input_path, sheet_name="Hourly Profile")  # read_excel(): read just the data sheet (not Instructions)

    required_cols = ["Date", "Hour", "Load_kWh"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        return {"hourly_profile": None, "errors": [f"Missing column(s): {missing}"]}

    if len(df) != 8760:
        errors.append(f"Expected 8,760 rows (365 days x 24 hours), found {len(df)}")
    if df["Load_kWh"].isna().any():
        n_missing = int(df["Load_kWh"].isna().sum())
        errors.append(f"{n_missing} row(s) have a blank Load_kWh value — every hour must be filled in")
    if (df["Load_kWh"].dropna() < 0).any():
        errors.append("Some Load_kWh value(s) are negative, which isn't physically valid")
    if not df["Hour"].between(0, 23).all():
        errors.append("Hour must be between 0 and 23 for every row")

    date_hour_pairs = df["Date"].astype(str) + "_" + df["Hour"].astype(str)  # build a combined key to check for duplicate/missing hour slots
    if date_hour_pairs.duplicated().any():
        errors.append("Some Date+Hour combination(s) appear more than once")

    if errors:
        return {"hourly_profile": None, "errors": errors}

    df["total_wh"] = df["Load_kWh"] * 1000  # convert kWh (the user-facing unit) to Wh (this project's internal working unit)
    return {"hourly_profile": df, "errors": []}

### Round-trip test

Export a blank template, fill it in with our already-computed profile (§2.4) standing in for "real user data", re-import it, and confirm the annual total matches — proving the upload path is lossless and produces a profile usable downstream.

In [24]:
hourly_template_path = DATA_DIR / "Hourly_Load_Profile_Template.xlsx"
export_hourly_profile_template(hourly_template_path)  # call: write the blank template
print("Exported blank template to:", hourly_template_path.resolve())

# Simulate a user filling it in, using our computed profile's values (a stand-in for real metered data)
filled_in = pd.read_excel(hourly_template_path, sheet_name="Hourly Profile")  # read the blank template back in
filled_in["Load_kWh"] = hourly_profile["total_wh"].values / 1000  # fill in Load_kWh from our computed profile, converted Wh -> kWh
filled_in.to_excel(hourly_template_path, sheet_name="Hourly Profile", index=False)  # to_excel(): overwrite just the data — good enough for this round-trip test

reimported = import_hourly_profile(hourly_template_path)  # call: read the "filled in" file back in through the real importer
if reimported["errors"]:
    print("Validation errors:", reimported["errors"])
else:
    reimported_total_gwh = reimported["hourly_profile"]["total_wh"].sum() / 1e9
    print(f"Re-imported annual total: {reimported_total_gwh:.6f} GWh")
    assert abs(reimported_total_gwh - annual_total_gwh) < 1e-6, "Round-trip mismatch!"
    print("✓ Round-trip export -> fill in -> re-import -> matches the original computed total")

Exported blank template to: /home/claude/rural-electrification/data/Hourly_Load_Profile_Template.xlsx


Re-imported annual total: 3.437285 GWh
✓ Round-trip export -> fill in -> re-import -> matches the original computed total


### Validation check (deliberately incomplete upload)

Negative test: leave some hours blank, and confirm the importer catches it rather than silently treating a blank as zero demand.

In [25]:
bad_profile_path = DATA_DIR / "_test_incomplete_profile.xlsx"
export_hourly_profile_template(bad_profile_path)  # start from a fresh blank template (all Load_kWh blank)

bad_result = import_hourly_profile(bad_profile_path)  # try to import it as-is, with every hour still blank
print("Errors detected:", bad_result["errors"])
assert bad_result["errors"], "Expected validation to catch the missing values but it didn't!"
print("✓ Validation correctly rejected the incomplete file")

bad_profile_path.unlink()  # clean up the test file

Errors detected: ['8760 row(s) have a blank Load_kWh value — every hour must be filled in']
✓ Validation correctly rejected the incomplete file


## 2.8 Summary

- Full 8,760-hour annual demand profile computed from Step 1's appliance data + the 19 day-type patterns + the corrected 365-day calendar — **3.437 GWh/year**, cross-checked three independent ways (exact match on an unaffected hour, an explained difference on the corrected total, and an independent hand-calculation on a corrected hour).
- Two editable-data pathways, both tested round-trip: the day-type "Qty Active" table (download, edit in Excel, re-upload), and a complete custom hourly profile (download a blank Date/Hour/Load template, fill in real data, upload — bypassing the day-type computation entirely).
- Everything is saved to `data/`: `default_daytype_profiles.csv`, `annual_calendar_2026.csv`, `hourly_load_profile_2026.csv`, plus the two Excel templates.

**Next notebook (Step 3):** visualize this profile — weekly, monthly, and yearly views, with switchable units (kWh/MWh/TWh).

## 2.9 Making the calendar rules user-editable

Everything above (§2.3) reproduced the source workbook's calendar *exactly as implemented*, patched only where it was clearly a copy-paste bug. That's the right way to **derive** correct defaults, but it isn't editable — the season boundaries and holiday/festival dates are baked into which Annex-III cells reference which Annex-II rows.

Per the user's request, three things need to become directly editable in the app:
1. **Season durations** — the start/end date of each season/period.
2. **Public holiday dates** — entered individually, not as a date range.
3. **Festival dates** — entered individually, not as a date range.

So we now rebuild the calendar as a small, transparent **rule-based generator** driven entirely by three editable tables, instead of reading Annex-III's formulas. We derive its default season boundaries directly from the already-validated §2.3 result (so the defaults reproduce exactly what we already proved correct), then confirm the two mechanisms agree on every one of the 365 days before switching over to using the editable version for the rest of this project.

In [26]:
# Default season periods: one row per named period, covering the full year exactly once (no gaps or overlaps).
# start/end are "MM-DD" (no year, so the same table works for any year). weekday_daytype applies Mon-Fri,
# weekend_daytype applies Sat-Sun. Derived from the validated §2.3 calendar (each boundary here is the exact
# date where §2.3's corrected day_type column changes).
default_season_periods = pd.DataFrame([
    {"period_name": "January",        "start_mmdd": "01-01", "end_mmdd": "01-31", "weekday_daytype": "January",        "weekend_daytype": "Winter_Weekend"},
    {"period_name": "February",       "start_mmdd": "02-01", "end_mmdd": "02-28", "weekday_daytype": "February",       "weekend_daytype": "Winter_Weekend"},
    {"period_name": "March",          "start_mmdd": "03-01", "end_mmdd": "03-31", "weekday_daytype": "March",          "weekend_daytype": "Winter_Weekend"},
    {"period_name": "April",          "start_mmdd": "04-01", "end_mmdd": "04-30", "weekday_daytype": "April",          "weekend_daytype": "Summer_Weekend"},
    {"period_name": "May",            "start_mmdd": "05-01", "end_mmdd": "05-31", "weekday_daytype": "May",            "weekend_daytype": "Summer_Weekend"},
    {"period_name": "June_1_15",      "start_mmdd": "06-01", "end_mmdd": "06-15", "weekday_daytype": "June_1_15",      "weekend_daytype": "Summer_Weekend"},
    {"period_name": "Early_Monsoon",  "start_mmdd": "06-16", "end_mmdd": "07-31", "weekday_daytype": "Early_Monsoon",  "weekend_daytype": "Early_Monsoon_Weekend"},
    {"period_name": "Peak_Monsoon_1", "start_mmdd": "08-01", "end_mmdd": "08-15", "weekday_daytype": "Peak_Monsoon_1", "weekend_daytype": "Peak_Monsoon_Weekend"},
    {"period_name": "Peak_Monsoon_2", "start_mmdd": "08-16", "end_mmdd": "09-15", "weekday_daytype": "Peak_Monsoon_2", "weekend_daytype": "Peak_Monsoon_Weekend"},
    {"period_name": "Late_Summer",    "start_mmdd": "09-16", "end_mmdd": "10-31", "weekday_daytype": "Late_Summer",    "weekend_daytype": "Late_Summer_Weekend"},
    {"period_name": "November",       "start_mmdd": "11-01", "end_mmdd": "11-30", "weekday_daytype": "November",       "weekend_daytype": "Winter_Weekend"},
    {"period_name": "December",       "start_mmdd": "12-01", "end_mmdd": "12-31", "weekday_daytype": "December",       "weekend_daytype": "Winter_Weekend"},
])
default_season_periods

,period_name,start_mmdd,end_mmdd,weekday_daytype,weekend_daytype
0,January,01-01,01-31,January,Winter_Weekend
1,February,02-01,02-28,February,Winter_Weekend
2,March,03-01,03-31,March,Winter_Weekend
3,April,04-01,04-30,April,Summer_Weekend
4,May,05-01,05-31,May,Summer_Weekend
5,June_1_15,06-01,06-15,June_1_15,Summer_Weekend
6,Early_Monsoon,06-16,07-31,Early_Monsoon,Early_Monsoon_Weekend
7,Peak_Monsoon_1,08-01,08-15,Peak_Monsoon_1,Peak_Monsoon_Weekend
8,Peak_Monsoon_2,08-16,09-15,Peak_Monsoon_2,Peak_Monsoon_Weekend
9,Late_Summer,09-16,10-31,Late_Summer,Late_Summer_Weekend


In [27]:
# Default public holidays and festival dates — each a single specific date, listed individually (year 2026),
# rather than a date range. These override the season/weekday-weekend assignment above on their exact date.
# (Includes the Jan 15 -> Jan 16 fix from §2.3.)
default_public_holidays = pd.DataFrame([
    {"date_mmdd": "01-16", "label": "Public Holiday"},
    {"date_mmdd": "02-05", "label": "Public Holiday"},
    {"date_mmdd": "04-04", "label": "Public Holiday"},
    {"date_mmdd": "05-01", "label": "Public Holiday"},
    {"date_mmdd": "08-14", "label": "Public Holiday"},
    {"date_mmdd": "11-09", "label": "Public Holiday"},
    {"date_mmdd": "12-25", "label": "Public Holiday"},
])

default_festival_holidays = pd.DataFrame([
    {"date_mmdd": "03-21", "label": "Festival"}, {"date_mmdd": "03-22", "label": "Festival"}, {"date_mmdd": "03-23", "label": "Festival"},
    {"date_mmdd": "05-28", "label": "Festival"}, {"date_mmdd": "05-29", "label": "Festival"},
    {"date_mmdd": "06-25", "label": "Festival"}, {"date_mmdd": "06-26", "label": "Festival"},
    {"date_mmdd": "08-25", "label": "Festival"},
])
print(f"{len(default_public_holidays)} public holidays, {len(default_festival_holidays)} festival days")

7 public holidays, 8 festival days


### The calendar generator

Given a year and the three editable tables above, this produces the same shape of output as §2.3's `corrected_calendar` — one row per day, with its final `day_type`. It also validates the season periods cover the whole year exactly once, since that's essential for correctness once these boundaries are user-editable (a careless edit could easily leave a gap or an overlap).

In [28]:
def validate_season_periods(season_periods_df: pd.DataFrame, year: int = 2026) -> list:
    """Check that the season periods cover every day of the year exactly once (no gaps, no overlaps). Returns a list of error strings (empty = OK)."""
    errors = []
    dates = pd.date_range(f"{year}-01-01", periods=365, freq="D")  # every calendar day of the year
    mmdd = dates.strftime("%m-%d")  # each date as "MM-DD", for comparing against the period boundaries
    covered_count = pd.Series(0, index=dates)  # how many periods claim each date — should end up exactly 1 everywhere

    for _, period in season_periods_df.iterrows():  # for loop: check each period's date range against every day
        in_period = (mmdd >= period["start_mmdd"]) & (mmdd <= period["end_mmdd"])  # boolean mask: which days fall in this period
        covered_count[in_period] += 1

    gaps = covered_count[covered_count == 0]  # days no period covers
    overlaps = covered_count[covered_count > 1]  # days more than one period covers
    if len(gaps) > 0:
        errors.append(f"{len(gaps)} day(s) not covered by any season period (e.g. {gaps.index[0].strftime('%b %d')})")
    if len(overlaps) > 0:
        errors.append(f"{len(overlaps)} day(s) covered by more than one season period (e.g. {overlaps.index[0].strftime('%b %d')})")
    return errors


def build_calendar_from_rules(year: int,
                               season_periods_df: pd.DataFrame,
                               public_holidays_df: pd.DataFrame,
                               festival_holidays_df: pd.DataFrame) -> pd.DataFrame:
    """Build the 365-day calendar (day type per day) from editable season/holiday/festival rules."""
    period_errors = validate_season_periods(season_periods_df, year)  # call: check the periods are well-formed before using them
    if period_errors:
        raise ValueError("Invalid season periods: " + "; ".join(period_errors))

    holiday_set = set(public_holidays_df["date_mmdd"])  # set(): fast membership testing for "is this date a holiday?"
    festival_set = set(festival_holidays_df["date_mmdd"])

    records = []  # collect one dict per day
    for date in pd.date_range(f"{year}-01-01", periods=365, freq="D"):  # for loop: walk through every day of the year
        mmdd = date.strftime("%m-%d")
        is_weekend = date.weekday() >= 5  # weekday(): Monday=0 ... Sunday=6, so 5 and 6 are Saturday/Sunday

        if mmdd in holiday_set:  # holidays/festivals override the season/weekday-weekend rule on their exact date
            day_type = "Public_Holiday"
        elif mmdd in festival_set:
            day_type = "Festival_Holiday"
        else:
            period = season_periods_df[(season_periods_df["start_mmdd"] <= mmdd) & (season_periods_df["end_mmdd"] >= mmdd)].iloc[0]  # find the one period this date falls in
            day_type = period["weekend_daytype"] if is_weekend else period["weekday_daytype"]

        records.append({"day": date.dayofyear, "date": date.date(), "weekday": date.strftime("%A"), "day_type": day_type})

    return pd.DataFrame(records)


editable_calendar = build_calendar_from_rules(2026, default_season_periods, default_public_holidays, default_festival_holidays)  # call: build the calendar from the editable defaults
print(f"Built calendar for {len(editable_calendar)} days")
editable_calendar.head(3)

Built calendar for 365 days


,day,date,weekday,day_type
0,1,2026-01-01,Thursday,January
1,2,2026-01-02,Friday,January
2,3,2026-01-03,Saturday,Winter_Weekend


### Cross-checking against the already-validated §2.3 calendar

The rule-based calendar's defaults were derived *from* the corrected §2.3 result, so if the derivation was done correctly, the two should agree on all 365 days — this is a strong end-to-end check that the new, editable mechanism hasn't silently changed anything.

In [29]:
# The two mechanisms should agree almost everywhere, since the rule-based defaults were derived FROM
# the corrected calendar. Where they don't agree, each mismatch needs to be understood individually
# before this new, editable mechanism can be trusted to replace the as-implemented one.
comparison = corrected_calendar[["date", "day_type"]].merge(  # merge(): line up both calendars by date for a direct comparison
    editable_calendar[["date", "day_type"]], on="date", suffixes=("_fixed", "_rulebased")
)
mismatches = comparison[comparison["day_type_fixed"] != comparison["day_type_rulebased"]]  # rows where the two mechanisms disagree

print(f"{len(mismatches)} mismatch(es) out of {len(comparison)} days")
display(mismatches)

4 mismatch(es) out of 365 days


,date,day_type_fixed,day_type_rulebased
198,2026-07-18,Peak_Monsoon_Weekend,Early_Monsoon_Weekend
199,2026-07-19,Peak_Monsoon_Weekend,Early_Monsoon_Weekend
205,2026-07-25,Peak_Monsoon_Weekend,Early_Monsoon_Weekend
206,2026-07-26,Peak_Monsoon_Weekend,Early_Monsoon_Weekend


**Investigation.** All 4 mismatches are weekends — Sat Jul 18, Sun Jul 19, Sat Jul 25, Sun Jul 26 — where the as-implemented workbook (`corrected_calendar`) already says `Peak_Monsoon_Weekend`, but the rule-based calendar says `Early_Monsoon_Weekend`.

The root cause: the source workbook's real, non-buggy behaviour is **asymmetric between weekdays and weekends** during this transition. Weekends switch from the Early-Monsoon pattern to the Peak-Monsoon pattern starting **July 18**, but weekdays keep using the Early-Monsoon pattern all the way through **July 31** (the boundary used for the `Early_Monsoon` period above). This isn't the April copy-paste bug from §2.3 — it's a genuine, separate detail of how the original workbook was built, only visible now because the rule-based model was cross-checked day-by-day against it.

A single start/end date range per named season — which is exactly what was asked for ("edit the season durations, starting and ending dates") — cannot represent a weekday transition date that differs from the weekend transition date within the same season. Two ways to handle it:

- **(a)** Give weekday and weekend independent date ranges per period. Matches the source exactly, but adds a second start/end pair to every one of the 12 periods for a discrepancy that affects only 4 days a year, and doesn't match what was actually requested (one date range per season).
- **(b)** Keep the simpler single-range-per-season model as requested, and accept that 4 weekend days a year (the width of one transition, right at a season boundary) use the previous season's weekend pattern instead of the new one.

**Decision: (b).** This keeps the calendar exactly as editable as requested, at the cost of a documented ~4-day/year edge case that's now called out explicitly (and easy to revisit later as an independent weekday/weekend range if it ever matters). Everything else — all 361 other days of the year, every season boundary, every holiday and festival — matches the as-implemented workbook exactly. The next cell locks this in as a guard: if a future edit ever introduces a *different* or *additional* mismatch, it will fail loudly rather than being silently absorbed.

In [30]:
# Guard: confirm the mismatches are EXACTLY the 4 documented weekend transition-date days, nothing more/fewer/different.
# This protects against a future edit to the default tables silently introducing a new, unnoticed discrepancy.
KNOWN_MISMATCH_DATES = {  # the 4 accepted mismatch dates, as explained above
    pd.Timestamp("2026-07-18").date(), pd.Timestamp("2026-07-19").date(),
    pd.Timestamp("2026-07-25").date(), pd.Timestamp("2026-07-26").date(),
}
mismatch_dates = set(mismatches["date"])  # set(): the dates the comparison actually found

assert mismatch_dates == KNOWN_MISMATCH_DATES, (
    f"Calendar mismatches changed from the documented set! Found {mismatch_dates}, expected {KNOWN_MISMATCH_DATES}. "
    "Investigate before trusting the rule-based calendar."
)
print("Confirmed: the only differences between the two calendars are the 4 documented weekend transition-date days")
print("(Jul 18, 19, 25, 26) — every other one of the 365 days matches exactly.")

Confirmed: the only differences between the two calendars are the 4 documented weekend transition-date days
(Jul 18, 19, 25, 26) — every other one of the 365 days matches exactly.


### Switching over: `editable_calendar` becomes the default calendar

Now that the rule-based calendar is validated (with the one documented, accepted exception above), it — not the as-implemented `corrected_calendar` — becomes the **default calendar the app actually uses**, since it's the only one built from user-editable rules. We recompute the hourly demand profile from it, compare the annual total against the §2.4 result to confirm the switch changes almost nothing (as expected — 4 weekend days out of 365), and then re-save `data/annual_calendar_2026.csv` and `data/hourly_load_profile_2026.csv` from the editable version so the rest of the project (and the app) builds on the editable path from here on.

In [31]:
# Rebuild the hour skeleton using the editable calendar instead of the as-implemented one, then recompute demand.
hour_skeleton_editable = build_hour_skeleton()  # call: same 8,760-row skeleton as before
hour_skeleton_editable = hour_skeleton_editable.merge(editable_calendar[["day", "date", "day_type"]], on="day", how="left")  # merge(): attach date + day_type from the rule-based calendar this time

hourly_profile_editable = compute_hourly_demand(hour_skeleton_editable, daytype_profiles, power_lookup, household_categories)  # call: same computation function as §2.4, new calendar input

original_total_gwh = hourly_profile["total_wh"].sum() / 1e9  # convert Wh -> GWh: from §2.4 (corrected_calendar)
editable_total_gwh = hourly_profile_editable["total_wh"].sum() / 1e9  # same, from the new editable_calendar

print(f"Annual total, corrected_calendar (as-implemented, §2.4): {original_total_gwh:.6f} GWh")
print(f"Annual total, editable_calendar  (rule-based, §2.9):     {editable_total_gwh:.6f} GWh")
print(f"Difference: {(editable_total_gwh - original_total_gwh) * 1000:.3f} MWh ({(editable_total_gwh / original_total_gwh - 1) * 100:+.4f}%)")
print("Small and expected — confined to the 4 documented weekend days switching from Peak-Monsoon to Early-Monsoon weekend patterns.")

Annual total, corrected_calendar (as-implemented, §2.4): 3.437285 GWh
Annual total, editable_calendar  (rule-based, §2.9):     3.440338 GWh
Difference: 3.052 MWh (+0.0888%)
Small and expected — confined to the 4 documented weekend days switching from Peak-Monsoon to Early-Monsoon weekend patterns.


In [32]:
# From here on, the editable calendar is the project's default — re-save the two files that depend on it.
editable_calendar.to_csv(DATA_DIR / "annual_calendar_2026.csv", index=False)  # to_csv(): overwrite §2.3's as-implemented version with the editable one
hourly_profile_editable.to_csv(DATA_DIR / "hourly_load_profile_2026.csv", index=False)  # overwrite §2.5's version with the one built on the editable calendar
print("Re-saved data/annual_calendar_2026.csv and data/hourly_load_profile_2026.csv from the editable calendar.")

Re-saved data/annual_calendar_2026.csv and data/hourly_load_profile_2026.csv from the editable calendar.


### Excel download / edit / re-upload for the calendar rules

Same pattern as every other editable dataset in this project (Step 1's appliance list, §2.6's day-type table, §2.7's hourly profile): download a formatted template, edit it in Excel, re-upload it, with validation before it's accepted. Here the template has three sheets — **Season Periods**, **Public Holidays**, **Festival Holidays** — matching the three editable tables above.

In [33]:
def export_calendar_rules_template(season_periods_df: pd.DataFrame, public_holidays_df: pd.DataFrame,
                                     festival_holidays_df: pd.DataFrame, output_path) -> None:
    """Write the three editable calendar tables (season periods, public holidays, festival holidays) to a formatted .xlsx."""
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:  # ExcelWriter(): open a new .xlsx for writing, with formatting support
        workbook = writer.book
        header_fmt = workbook.add_format({"bold": True, "bg_color": "#2a78d6", "font_color": "white", "border": 1})  # same header style used throughout this project

        instructions = pd.DataFrame({"Instructions": [
            "Season Periods: each row is a named period of the year, with a start and end date (MM-DD, no year — the same table applies every year).",
            "Periods must cover all 365 days exactly once, with no gaps and no overlaps. Edit start_mmdd/end_mmdd to change season durations.",
            "weekday_daytype / weekend_daytype select which usage pattern (from the Day Type Profiles table) applies on weekdays / weekends within that period.",
            "Public Holidays / Festival Holidays: each row is ONE specific date (MM-DD). Add or remove rows to add/remove individual holiday or festival dates.",
            "A date listed as a holiday or festival always overrides the season/weekday-weekend pattern on that exact date.",
            "Save the file, then re-upload it in the app to use these values instead of the defaults.",
        ]})
        instructions.to_excel(writer, sheet_name="Instructions", index=False)
        season_periods_df.to_excel(writer, sheet_name="Season Periods", index=False)  # to_excel(): write each table to its own sheet
        public_holidays_df.to_excel(writer, sheet_name="Public Holidays", index=False)
        festival_holidays_df.to_excel(writer, sheet_name="Festival Holidays", index=False)

        sheet_frames = [("Instructions", instructions), ("Season Periods", season_periods_df),  # for loop: bold headers + sized columns on every sheet
                         ("Public Holidays", public_holidays_df), ("Festival Holidays", festival_holidays_df)]
        for sheet_name, df in sheet_frames:
            ws = writer.sheets[sheet_name]
            for col_idx, col_name in enumerate(df.columns):
                ws.write(0, col_idx, col_name, header_fmt)
                width = max(10, min(30, len(str(col_name)) + 2))
                ws.set_column(col_idx, col_idx, width)


export_calendar_rules_template(default_season_periods, default_public_holidays, default_festival_holidays,
                                DATA_DIR / "Calendar_Rules_Template.xlsx")  # call: write the default rules out as an editable template
print("Saved to:", (DATA_DIR / "Calendar_Rules_Template.xlsx").resolve())

Saved to: /home/claude/rural-electrification/data/Calendar_Rules_Template.xlsx


In [34]:
_MMDD_RE = re.compile(r"^\d{2}-\d{2}$")  # compile(): MM-DD format check, reused for every date column below


def _valid_mmdd_series(series: pd.Series) -> pd.Series:
    """Boolean mask: which values in a Series look like a valid MM-DD date string."""
    format_ok = series.astype(str).str.match(_MMDD_RE)  # match(): does it look like "MM-DD" at all?
    parsed = pd.to_datetime("2026-" + series.astype(str), format="%Y-%m-%d", errors="coerce")  # to_datetime(): also rejects real-calendar nonsense like 02-30
    return format_ok & parsed.notna()


def import_calendar_rules_template(input_path) -> dict:
    """Read a workbook in the export_calendar_rules_template() shape back into the three tables, with validation."""
    errors = []
    sheets = pd.read_excel(input_path, sheet_name=None)  # read_excel(): read every sheet at once

    required_sheets = ["Season Periods", "Public Holidays", "Festival Holidays"]
    missing_sheets = [s for s in required_sheets if s not in sheets]
    if missing_sheets:
        return {"season_periods": None, "public_holidays": None, "festival_holidays": None,
                "errors": [f"Missing required sheet(s): {missing_sheets}"]}

    season_periods = sheets["Season Periods"]
    public_holidays = sheets["Public Holidays"]
    festival_holidays = sheets["Festival Holidays"]

    required_cols = {  # required column(s) per sheet
        "Season Periods": ["period_name", "start_mmdd", "end_mmdd", "weekday_daytype", "weekend_daytype"],
        "Public Holidays": ["date_mmdd", "label"],
        "Festival Holidays": ["date_mmdd", "label"],
    }
    for sheet_name, df in [("Season Periods", season_periods), ("Public Holidays", public_holidays), ("Festival Holidays", festival_holidays)]:
        missing = [c for c in required_cols[sheet_name] if c not in df.columns]
        if missing:
            errors.append(f"'{sheet_name}' sheet is missing column(s): {missing}")

    if not errors:
        # Date-format checks
        if not _valid_mmdd_series(season_periods["start_mmdd"]).all() or not _valid_mmdd_series(season_periods["end_mmdd"]).all():
            errors.append("Season Periods: start_mmdd/end_mmdd must all be valid MM-DD dates")
        for sheet_name, df in [("Public Holidays", public_holidays), ("Festival Holidays", festival_holidays)]:
            if not _valid_mmdd_series(df["date_mmdd"]).all():
                errors.append(f"{sheet_name}: date_mmdd must all be valid MM-DD dates")
            if df["date_mmdd"].duplicated().any():  # duplicated(): flag the same date listed twice within one sheet
                errors.append(f"{sheet_name}: contains duplicate date(s)")

        # Season periods must cover the year exactly once (reuses the checker built earlier in this section)
        if not errors:
            period_errors = validate_season_periods(season_periods)  # call: same gap/overlap validator used by build_calendar_from_rules()
            errors.extend(period_errors)

        # A date shouldn't be listed as both a public holiday and a festival
        overlap = set(public_holidays["date_mmdd"]) & set(festival_holidays["date_mmdd"])  # set intersection: dates in both sheets
        if overlap:
            errors.append(f"Date(s) listed as both a Public Holiday and a Festival: {sorted(overlap)}")

    if errors:
        return {"season_periods": None, "public_holidays": None, "festival_holidays": None, "errors": errors}
    return {"season_periods": season_periods, "public_holidays": public_holidays, "festival_holidays": festival_holidays, "errors": []}

In [35]:
# Round-trip test: export the defaults, re-import them, confirm every table matches exactly.
imported = import_calendar_rules_template(DATA_DIR / "Calendar_Rules_Template.xlsx")  # call: read back the file just written
assert imported["errors"] == [], f"Round-trip import reported errors: {imported['errors']}"
pd.testing.assert_frame_equal(imported["season_periods"].reset_index(drop=True), default_season_periods.reset_index(drop=True))  # assert_frame_equal(): exact match, defaults -> file -> back
pd.testing.assert_frame_equal(imported["public_holidays"].reset_index(drop=True), default_public_holidays.reset_index(drop=True))
pd.testing.assert_frame_equal(imported["festival_holidays"].reset_index(drop=True), default_festival_holidays.reset_index(drop=True))
print("Round-trip OK: exported defaults re-imported with zero errors and an exact match on all three tables.")

# Negative test: a season-period edit that leaves a gap should be rejected, not silently accepted.
broken_periods = default_season_periods.copy()  # copy(): don't mutate the real defaults
broken_periods.loc[0, "end_mmdd"] = "01-20"  # shrink January's range without extending anything else -> a gap from Jan 21-31
broken_periods.to_excel(DATA_DIR / "_test_broken_calendar.xlsx", sheet_name="Season Periods", index=False)  # write just this one sheet
with pd.ExcelWriter(DATA_DIR / "_test_broken_calendar.xlsx", engine="xlsxwriter") as writer:  # ExcelWriter(): rewrite with all required sheets this time
    broken_periods.to_excel(writer, sheet_name="Season Periods", index=False)
    default_public_holidays.to_excel(writer, sheet_name="Public Holidays", index=False)
    default_festival_holidays.to_excel(writer, sheet_name="Festival Holidays", index=False)
broken_result = import_calendar_rules_template(DATA_DIR / "_test_broken_calendar.xlsx")
assert broken_result["season_periods"] is None and len(broken_result["errors"]) > 0, "A calendar with a date gap should be rejected"
print(f"Negative test OK: a season-period gap was correctly rejected ({broken_result['errors'][0]})")
(DATA_DIR / "_test_broken_calendar.xlsx").unlink()  # unlink(): remove the scratch test file, keep data/ clean

Round-trip OK: exported defaults re-imported with zero errors and an exact match on all three tables.


Negative test OK: a season-period gap was correctly rejected (11 day(s) not covered by any season period (e.g. Jan 21))


In [36]:
# Save the three default calendar-rule tables as CSV too (consistent with every other default dataset in data/).
default_season_periods.to_csv(DATA_DIR / "default_season_periods.csv", index=False)
default_public_holidays.to_csv(DATA_DIR / "default_public_holidays.csv", index=False)
default_festival_holidays.to_csv(DATA_DIR / "default_festival_holidays.csv", index=False)
print("Saved default_season_periods.csv, default_public_holidays.csv, default_festival_holidays.csv to data/")

Saved default_season_periods.csv, default_public_holidays.csv, default_festival_holidays.csv to data/


## 2.10 Summary (updated)

- The calendar is now driven by three small, fully editable tables — season periods (start/end date, weekday & weekend pattern), public holidays, and festival dates — instead of being hard-derived from the source workbook's formulas. Every one of the 365 days matches the as-implemented, bug-fixed §2.3 calendar exactly, **except 4 documented weekend days (Jul 18/19/25/26)** where a weekday/weekend-asymmetric transition in the source data can't be represented by a single date range per season; this was investigated, quantified (+0.089% on the annual total), and accepted as a deliberate simplification rather than silently absorbed.
- `editable_calendar` (and the hourly profile computed from it) is now the project's default going forward — `data/annual_calendar_2026.csv` and `data/hourly_load_profile_2026.csv` were re-saved from it.
- Excel download/edit/re-upload built and tested for the three calendar-rule tables (`export_calendar_rules_template()` / `import_calendar_rules_template()`), including validation that edited season periods still cover the year with no gaps or overlaps, and a negative test confirming a broken edit is rejected. Template file: `data/Calendar_Rules_Template.xlsx`.
- New files in `data/`: `default_season_periods.csv`, `default_public_holidays.csv`, `default_festival_holidays.csv`, `Calendar_Rules_Template.xlsx`.

**Step 2 (Hourly Load Profile of the Year) is now fully complete, including user-editable season/holiday/festival dates.**